# Stage 1 본 계산 — Colab에서 돌리기

노트북 대신 Colab에서 `stage1_g1.py`를 돌린다. **결과는 Google Drive에 과제마다 저장**되므로 세션이 끊겨도 다시 연결해 1번 셀부터 실행하면 끝난 과제는 건너뛰고 이어서 한다.

- 무료 Colab은 CPU가 보통 2개라 `WORKERS = 2`
- 코드는 GitHub의 `stage1` 브랜치에서 받는다 (먼저 push 필요)
- 노트북과 동시에 돌릴 때는 **서로 다른 묶음**을 맡긴다 (같은 과제를 둘이 하면 시간 낭비)


In [ ]:
# 1. Google Drive 연결 (결과 저장용)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 코드 받기 (stage1 브랜치)
REPO_URL = "https://github.com/Kooseoyoon/ssbt_battery_diagnosis.git"
REPO_DIR = "/content/ssbt_battery_diagnosis"
BRANCH = "stage1"
import os
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch -q && git checkout -q {BRANCH} && git pull -q
!cd {REPO_DIR} && git log --oneline -3

In [ ]:
# 3. 경로 설정: 데이터는 저장소 안(data/processed), 결과는 Drive
RESULTS_ON_DRIVE = "/content/drive/MyDrive/SSBT_battery_diagnosis/results"
os.environ["SSBT_DATA_DIR"] = f"{REPO_DIR}/data"
os.environ["SSBT_RESULTS_DIR"] = RESULTS_ON_DRIVE
os.environ["PYTHONUTF8"] = "1"
os.makedirs(RESULTS_ON_DRIVE, exist_ok=True)
print("CPU:", os.cpu_count())
!ls {REPO_DIR}/data/processed | head

In [ ]:
# 4. 테스트 (1분 이내). 전부 passed여야 계산을 시작한다
!pip -q install pytest
!cd {REPO_DIR} && python -m pytest tests -q 2>&1 | tail -2

In [ ]:
# 5. 본 계산 — 맡은 묶음 하나만 남기고 실행 (끊기면 이 셀을 다시 실행)
WORKERS = 2
JOB = "--variant monotone --inv 64 --eol 60"                # 묶음 D (통째로)
# JOB = "--variant excl --inv 64 --eol 60 --shard 1/2"      # 묶음 E의 절반 (노트북은 0/2)
# JOB = "--inv 32 64 --eol 55"                              # 묶음 B
# JOB = "--inv 32 64 --eol 65"                              # 묶음 C
!cd {REPO_DIR} && python src/stage1_g1.py run {JOB} --workers {WORKERS}

In [ ]:
# 6. (선택) 결과 표 만들기 — variant 에 맞춰서
!cd {REPO_DIR} && python src/stage1_g1.py finalize --variant monotone
!ls {RESULTS_ON_DRIVE}/stage1